In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Custom modül
import sys
sys.path.append('../')
from src.model_training import ModelTrainer

print("✅ Kütüphaneler yüklendi")

In [ ]:
from pathlib import Path

# Tek dosyadan okuma
DATA_FILE = Path(r'C:\Users\PC\Desktop\creditwise\data\processed\train_processed.csv')

df = pd.read_csv(DATA_FILE)

# Hedef ve grup sütunlarını belirle
_target_candidates = ['Credit_Score', 'en']
target_col = next((c for c in _target_candidates if c in df.columns), None)
if target_col is None:
    raise ValueError("Hedef sütun bulunamadı. 'Credit_Score' ya da 'en' bekleniyor.")

group_col = 'Customer_ID' if 'Customer_ID' in df.columns else None
if group_col is None:
    raise ValueError("Customer_ID sütunu bulunamadı.")

# Ayrıştırma
y_train = df[target_col]
customer_ids = df[group_col]
X_train = df.drop(columns=[target_col, group_col], errors='ignore')

print("Veri boyutları:")
print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"Customer IDs (unique): {customer_ids.nunique()}")

In [ ]:
# Hedef değişken dağılımı
plt.figure(figsize=(10, 6))

plt.subplot(1, 2, 1)
y_train.value_counts().plot(kind='bar')
plt.title('Credit Score Dağılımı')
plt.xlabel('Credit Score')
plt.ylabel('Frekans')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
y_train.value_counts(normalize=True).plot(kind='pie', autopct='%1.1f%%')
plt.title('Credit Score Oranları')

plt.tight_layout()
plt.show()

print("Hedef değişken dağılımı:")
print(y_train.value_counts())

In [ ]:
# Model trainer nesnesini oluştur
trainer = ModelTrainer(random_state=42)
print("✅ Model trainer oluşturuldu")

In [ ]:
# Tüm modelleri eğit
print("🚀 Model eğitimi başlıyor...")
print("Bu işlem birkaç dakika sürebilir...")

models = trainer.train_all(
    X=X_train, 
    y=y_train, 
    groups=customer_ids,  # GroupKFold için
    cv_folds=5
)

print("✅ Tüm modeller eğitildi!")

In [ ]:
# Cross validation sonuçlarını görüntüle
cv_results = trainer.get_cv_results()
print("📊 Cross Validation Sonuçları:")
print("=" * 40)
print(cv_results)

# Görselleştirme
plt.figure(figsize=(10, 6))
cv_results['CV_Mean'].plot(kind='barh', xerr=cv_results['CV_Std'], 
                           capsize=4, color='skyblue', edgecolor='black')
plt.xlabel('F1 Score (Weighted)')
plt.title('Model Performance Karşılaştırması (5-Fold CV)')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Tüm modelleri kaydet
print("💾 Modeller kaydediliyor...")
trainer.save_models(model_dir="../models/")
print("✅ Tüm modeller kaydedildi!")

# Kaydedilen dosyaları listele
import os
model_files = os.listdir("../models/")
print("\nKaydedilen dosyalar:")
for file in sorted(model_files):
    print(f"  📁 {file}")

In [ ]:
# Model dosya boyutlarını kontrol et
import os

print("📏 Model Dosya Boyutları:")
print("-" * 30)

for file in sorted(os.listdir("../models/")):
    if file.endswith('.pkl'):
        file_path = os.path.join("../models/", file)
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f"{file:<25}: {size_mb:.2f} MB")

In [ ]:
# Eğitim özeti
print("=" * 60)
print("🎯 MODEL EĞİTİMİ TAMAMLANDI!")
print("=" * 60)

print(f"📊 Toplam Model Sayısı: {len(models)}")
print(f"📁 Kaydedilen Konum: ../models/")

print("\n🏆 Model Performansları:")
for idx, (model_name, score) in enumerate(cv_results.iterrows(), 1):
    print(f"{idx}. {model_name:<18}: {score['CV_Mean']:.4f} (±{score['CV_Std']:.4f})")

print("\n📋 Sonraki Adımlar:")
print("  1. 04_evaluation.ipynb notebook'unu çalıştır")
print("  2. Test seti performansını değerlendir")
print("  3. En iyi modeli seç")
print("  4. Model interpretability analizi yap")

print("=" * 60)

In [ ]:
# Memory cleanup
import gc

print("🧹 Memory cleanup...")
del models  # model referanslarını temizle
gc.collect()
print("✅ Cleanup tamamlandı")